# STEP 5.5 · JTSI 민감도 검토 (지수 방어막)

| 버전 | 구성 |
|---|---|
| v1 (기본) | 피로 + AI리스크 + 역할갭 + 번아웃 |
| v2 (단순) | 피로 + AI리스크 |
| v3 (확장) | v1 + 직무불만족(10-만족도) |

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
OUT_TABLES = ROOT / "outputs" / "tables"
OUT_FIGURES = ROOT / "outputs" / "figures"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_FIGURES.mkdir(parents=True, exist_ok=True)

import pyreadstat, pandas as pd, numpy as np
from scipy.stats import zscore, spearmanr
from src.variable_mapping import MEDIA_LABELS
import warnings; warnings.filterwarnings('ignore')
df25,_ = pyreadstat.read_sav(DATA_RAW / "journalist_2025.sav")
d=df25.copy()
d['fatigue']=d['q5']; d['ai_user']=(d['q38']==1)
d['risk']=d[['q39_4','q39_5','q39_6']].mean(1)
d['burnout']=d[['q21_3','q21_4']].mean(1); d['satis']=d['q19_1']
d['role_imp']=d[[f'q2_{i}' for i in range(1,8)]].mean(1)
d['role_perf']=d[[f'q3_{i}' for i in range(1,8)]].mean(1)
d['role_gap']=d['role_imp']-d['role_perf']
d25=d

In [2]:
dd = d25.dropna(subset=['fatigue','risk','role_gap','burnout','satis']).copy()
z = lambda x: zscore(x)
dd['v1'] = z(dd['fatigue'])+z(dd['risk'])+z(dd['role_gap'])+z(dd['burnout'])
dd['v2'] = z(dd['fatigue'])+z(dd['risk'])
dd['v3'] = dd['v1'] + z(10-dd['satis'])

In [3]:
# 매체별 평균 스트레스
rank = pd.DataFrame()
for v in ['v1','v2','v3']:
    rank[v] = dd.groupby('SQ1')[v].mean()
rank.index = [MEDIA_LABELS[int(i)] for i in rank.index]
rank.index.name = "매체유형"   # CSV 첫 컬럼명 지정
rank = rank.round(3)
rank.to_csv(OUT_TABLES / "jtsi_sensitivity.csv", encoding='utf-8-sig')
rank

,v1,v2,v3
매체유형,,,
신문사,0.063,0.137,0.117
방송사,0.120,-0.053,0.024
인터넷언론사,-0.441,-0.404,-0.520
뉴스통신사,0.245,0.059,0.273


In [4]:
# 버전 간 순위 상관 (Spearman) - 얼마나 일관적인지 수치로 확인
for a in ['v2','v3']:
    rho,_ = spearmanr(rank['v1'].rank(), rank[a].rank())
    print(f"v1 vs {a} 순위상관: {rho:.2f}")

v1 vs v2 순위상관: 0.40
v1 vs v3 순위상관: 0.80


### 해석 (★)

| 비교 | 순위상관 | 해석 |
|---|---|---|
| v1 vs v3 (확장) | **0.80** | 높음 — 만족도를 더해도 결론 거의 유지 |
| v1 vs v2 (단순) | **0.40** | 약함 — 2요소만 쓰면 순위가 흔들림 |

**결론**: 민감도는 '부분적으로' 안정적이다.
- **확고한 부분**: 인터넷언론사가 세 버전 **모두에서 최저**
  (스트레스 가장 낮음). 이 결론은 구성을 바꿔도 안 흔들린다.
- **주의할 부분**: 전통매체 3종(신문·방송·통신)의 **내부 순위**는
  버전에 따라 바뀐다. 특히 단순 모델(v2)에서 변동이 크다.

> 보고서 문장: *"인터넷언론사의 상대적 저위험은
> 지수 구성과 무관하게 일관되게 나타났다. 다만 전통매체 간
> 세부 순위는 구성에 따라 변동하므로, 개별 매체의 순위보다
> '전통매체 대 인터넷매체'라는 큰 구도로 해석하는 것이 타당하다."*
